# PANORAMA GERAL — Por que mudar o shaking?


Antes, o shaking usava operadores muito fracos (pequenas reversoes, 2-opt aleatorio, insercoes aleatorias, perturbacoes que duplicavam a LS). Eles mexiam pouco na estrutura do tour, eram facilmente desfeitos pela local search, criavam loops e nao ampliavam o basin of attraction.


Agora entraram 5 operadores modernos (os mesmos usados em LKH, ILS classico, GVNS, RVNS, HGLS e VNS de ponta). Sao perturbacoes estruturais que mudam de vale e trazem diversificacao real. 

# Operadores de shaking — passo a passo


## 1) Double-Bridge — perturbacao forte classica


- O que faz: remove 4 arestas nao adjacentes e reconecta em ordem cruzada, criando duas pontes novas.
- Por que e forte: nenhuma LS baseada em 2-opt/3-opt/insertion/swap reconstroi exatamente essas 4 arestas; leva o tour para outro vale e quebra estagnacao.
- Quando usar: para diversificar globalmente ou sair de otimos profundos em instancias medias/grandes.
- Efeito visual: preserva dois blocos internos, mas muda a macro-ordem deles (saltos grandes sem destruir tudo).


## 2) k-Exchange — remover k nos e reinserir embaralhados


- O que faz: escolhe k nos (tipicamente 3–10), retira como bloco, embaralha a ordem e reinsere no tour.
- Por que funciona: desmonta clusters locais e permite recombinar trechos que insertion/2-opt nunca explorariam.
- Quando usar: middle kick — mais forte que shuffle, menos destrutivo que double-bridge.
- Efeito visual: pequenos buracos abrem espaco para novas conexoes; bom para quebrar ciclos locais sem perder toda a estrutura.


## 3) Cross-Exchange — troca de segmentos longos


- O que faz: seleciona dois segmentos [A...B] e [C...D] e troca suas posicoes (mantendo cada segmento intacto).
- Por que funciona: realoca blocos inteiros, reorganiza clusters e altera macro-ordem de regioes sem caos total.
- Quando usar: tours que ja saturaram 2-opt/3-opt; bom para instancias com regioes bem definidas.
- Efeito visual: o desenho do tour fica parecido, mas as regioes trocam de lugar, abrindo novos vizinhos.


## 4) Block Recombine — recombinacao de blocos


- O que faz: particiona o tour em B blocos, embaralha apenas a ordem dos blocos e mantem cada bloco intacto.
- Por que funciona: atua como um crossover interno de GA — preserva subrotas boas e muda a ordem das ruins.
- Quando usar: instancias com clusters naturais; quer mexer na macroestrutura sem destruir microestruturas boas.
- Efeito visual: mesmas pecas, ordem diferente; combina diversificacao com preservacao do que ja funciona.


## 5) Shuffle Segment — embaralhar a ordem de um segmento


- O que faz: escolhe um segmento contiguo e permuta internamente os nos (nao e apenas reversao).
- Por que funciona: gera padroes de vizinhanca que OR-OPT/2-opt nao achariam; perturba leve, mas nao reversivel.
- Quando usar: diversificacao rapida e barata; boa opcao antes de uma LS forte que refine o novo arranjo.
- Efeito visual: o segmento muda de ordem, criando novas combinacoes locais sem destruir o tour inteiro.


## Resumo rapido (qual usar quando)


- Preciso de salto global: Double-Bridge.
- Quero desmontar e reconstruir clusters: k-Exchange.
- Quero trocar regioes inteiras: Cross-Exchange.
- Quero recombinar subrotas boas: Block Recombine.
- Quero perturbar pouco mas com variedade real: Shuffle Segment.

_shake_double_bridge

Criar uma perturbação forte removendo 4 segmentos e reconectando em outra ordem.


É o operador clássico de ILS (Helsgaun também usa em LKH).


🔍 Como funciona


Escolhe 4 cortes: a < b < c < d
Divide o tour em 5 segmentos:
S1 = [0..a), S2 = [a..b), S3 = [b..c), S4 = [c..d), S5 = [d..end)
Reconecta como:
S1 + S3 + S2 + S4 + S5

📌 Código demonstrativo

In [6]:
import numpy as np
import random
def shake_double_bridge(tour):
    core = tour[:-1]
    n = len(core)
    
    a,b,c,d = sorted(random.sample(range(1,n),4))
    print(f"Cuts at positions: {a}, {b}, {c}, {d}")
    S1 = core[:a]
    S2 = core[a:b]
    S3 = core[b:c]
    S4 = core[c:d]
    S5 = core[d:]
    
    print(S3)
    new_core = np.concatenate([S1, S3, S2, S4, S5])
    return np.append(new_core, new_core[0])

In [7]:
tour = np.array([0,1,2,3,4,5,6,7,8,9,0])
new_tour = shake_double_bridge(tour.copy())
print("Original tour:", tour)
print("New tour:", new_tour)

Cuts at positions: 1, 3, 4, 9
[3]
Original tour: [0 1 2 3 4 5 6 7 8 9 0]
New tour: [0 3 1 2 4 5 6 7 8 9 0]


2. _shake_k_exchange

🎯 Objetivo

Remover k nós aleatórios e reinserir em ordem embaralhada → perturbação média‑forte.

 Vantagem: é destrutivo o bastante para escapar de ótimos locais, sem virar random restart.
 
🔍 Como funciona

Seleciona k posições
Remove os nós
Reembaralha e reinsere como bloco
Fechamento do tour

In [10]:
def k_exchange_shake(tour, k):
    is_closed = tour[0] == tour[-1]
    core = tour[:-1] if is_closed else tour
    n = len(core)

    idxs = sorted(random.sample(range(1, n), k))
    removed_nodes = [core[i] for i in idxs]

    print(f"Removed nodes at positions: {idxs} -> {removed_nodes}")

    new_core = np.delete(core, idxs)
    random.shuffle(removed_nodes)

    insert_pos = random.randrange(1, len(new_core)+1)
    new_core = np.concatenate((new_core[:insert_pos], removed_nodes, new_core[insert_pos:]))

    if is_closed:
        new_core = np.concatenate([new_core, [new_core[0]]])

    return new_core

In [11]:
tour = np.array([0,1,2,3,4,5,6,7,8,9,0])
k = 3
new_tour_k_exchange = k_exchange_shake(tour.copy(), k)
print(f"\nK-Exchange Shake with k={k}:")
print("Original tour:", tour)
print("New tour:", new_tour_k_exchange)

Removed nodes at positions: [3, 5, 6] -> [np.int64(3), np.int64(5), np.int64(6)]

K-Exchange Shake with k=3:
Original tour: [0 1 2 3 4 5 6 7 8 9 0]
New tour: [0 1 2 5 6 3 4 7 8 9 0]


2. Segment-shuffle (embaralhar subrotas)
Escolhe uma subsequência relativamente longa e reordena internamente.
Diferente de 2-opt, 3-opt e flips, porque você faz permutação total, não reversão.

In [13]:
def shuffle_segment(tour, seg_len=6):
    is_closed = tour[0] == tour[-1]
    core = tour[:-1] if is_closed else tour
    n = len(core)

    start = random.randrange(1, n - seg_len)
    segment = core[start:start+seg_len].copy()
    random.shuffle(segment)

    print(f"Shuffling segment starting at {start} with length {seg_len}: {core[start:start+seg_len]} -> {segment}")

    new_core = np.concatenate([core[:start], segment, core[start+seg_len:]])
    if is_closed:
        new_core = np.concatenate([new_core, [new_core[0]]])
    return new_core

In [15]:
tour = np.array([0,1,2,3,4,5,6,7,8,9,0])
new_tour_shuffled_segment = shuffle_segment(tour.copy(), seg_len=4)
print(f"\nShuffle Segment Shake with segment length=4:")
print("Original tour:", tour)   
print("New tour:", new_tour_shuffled_segment)

Shuffling segment starting at 5 with length 4: [5 6 7 8] -> [6 7 8 5]

Shuffle Segment Shake with segment length=4:
Original tour: [0 1 2 3 4 5 6 7 8 9 0]
New tour: [0 1 2 3 4 6 7 8 5 9 0]


Cross‑exchange (2‑in‑2‑out)
Trocamos segmentos entre duas partes distintas da rota — tipo 2‑opt, mas envolvendo intervalos.

In [ ]:
def cross_exchange(tour):
    is_closed = tour[0] == tour[-1]
    core = tour[:-1] if is_closed else tour
    n = len(core)

    i, j = sorted(random.sample(range(1, n-1), 2))
    k, l = sorted(random.sample(range(1, n-1), 2))

    if i == k: 
        return tour

    s1 = core[i:j]
    s2 = core[k:l]

    new_core = core.copy()
    new_core[i:i+len(s2)] = s2
    new_core[k:k+len(s1)] = s1

    if is_closed:
        new_core = np.concatenate([new_core, [new_core[0]]])
    return new_core

In [ ]:
Or-opt extended (1‑2‑3 relocations)
Or-opt clássico, mas em shaking você permite:

mover blocos de tamanho 1
mover blocos de tamanho 2
mover blocos de tamanho 3
…mas em posições aleatórias grandes

Isto é muito mais destrutivo que sua inserção local, mas não repetitivo.

In [ ]:
def _shake_oropt_block(self, tour: np.ndarray, block_len: int = 2) -> np.ndarray:
    core, is_closed = self._extract_core(tour)
    n = len(core)
    if n < block_len + 3:
        return tour.copy()

    start = random.randrange(1, n - block_len + 1)
    block = core[start:start + block_len].copy()
    remainder = np.concatenate([core[:start], core[start + block_len:]])

    # Inserir “longe” do start: força de shaking
    far_min = max(1, start + block_len + max(2, n // 10))
    far_candidates = list(range(1, n - block_len + 1))  # [1..n-block_len]
    # Remover janela próxima ao start para evitar no-op
    far_candidates = [pos for pos in far_candidates if (pos < start - 1 or pos > far_min)]
    if not far_candidates:
        insert_pos = random.randrange(1, len(remainder) + 1)
    else:
        insert_pos = random.choice(far_candidates)
        if insert_pos > start:
            insert_pos -= block_len

    new_core = np.concatenate([remainder[:insert_pos], block, remainder[insert_pos:]])
    return self._reclose(new_core, is_closed)

## 🔍 Por que dividi assim?

### 🔵 k = 1 → vizinhanças leves (micro-perturbação)

Usei operadores que:

- mexem pouco  
- não destroem a estrutura do tour  
- não duplicam a busca local (LS)

**Operadores:**
- ✔ Or-opt 2  
- ✔ Or-opt 3  
- ✔ Shuffle pequeno  

💡 **Objetivo:** “sair da bacia local próxima”, mantendo quase toda a estrutura do tour.

---

### 🟣 k = 2 → vizinhanças moderadas (perturbação média)

Aqui usei operadores que mudam parte do tour, mas sem quebrá-lo inteiro:

**Operadores:**
- ✔ k-exchange leve (remove 3–4 nós)  
- ✔ cross-exchange curto  
- ✔ shuffle médio  

💡 **Objetivo:** explorar regiões mais distantes, mas ainda reconhecíveis pela LS.

---

### 🔴 k = 3 → vizinhanças fortes (perturbação alta)

Aqui já estamos falando de movimentos usados como *kick* em ILS:

**Operadores:**
- ✔ double-bridge (super clássico)  
- ✔ k-exchange médio  
- ✔ cross-exchange maior  
- ✔ block recombination  

💡 **Objetivo:** realmente sair de ótimos locais profundos.

---

### 🟢 k ≥ 4 → vizinhanças muito fortes

Nesta faixa, o *shaking* precisa:

- quebrar completamente a estrutura  
- produzir tours quase aleatórios  
- ainda manter alguns blocos estruturais  

**Operadores:**
- ✔ double-bridge  
- ✔ k-exchange de alto grau (5–8 nós)  
- ✔ cross-exchange forte  
- ✔ recombinação por blocos  
- ✔ shuffle grande  

💡 **Objetivo:** perturbar bastante — útil quando a busca fica estagnada.

---

## 🎯 Resumo fácil de entender

Em uma frase:

> **Cada k representa um nível de destruição diferente, compatível com a filosofia do VNS.**

- **k = 1** → movimentos pequenos (ajustes)  
- **k = 2** → movimentos médios (rearranjos)  
- **k = 3** → movimentos fortes (escape de ótimos locais)  
- **k ≥ 4** → movimentos muito fortes (quase um *restart*)

Essa estratificação segue diretamente o **VNS clássico**  
*(Hansen & Mladenović, 1997–2019).*

---

## 🧠 Por que isso funciona melhor do que sortear qualquer operador?

Simples:

- 🔸 Permite controle sistemático entre **intensificação vs diversificação**  
- 🔸 Evita duplicar vizinhanças com a busca local  
- 🔸 Dá **estabilidade estatística** ao VNS  
- 🔸 Implanta o racional *“shake leve → shake forte”*  
- 🔸 Aumenta o escape progressivo de ótimos locais  

Esse é exatamente o racional usado em:

- **GVNS**  
- **RVNS**  
- **BVNS**  
- **Skewed-VNS**  
- **VNS para TSP (Helsgaun / LKH)**  
- **VNS para VRP (Vidal)**  
- **VNS multi-neighborhood da literatura moderna**


## 🔧 Melhorias na Local Search

### 1️⃣ `limited_subsequence_flip` ≈ 2-opt disfarçado

Esse operador faz essencialmente:


➡️ Isso **já acontece no 2-opt e no 3-opt**.

**Resultado prático:**
- ✔ Operador duplicado  
- ✔ Não encontra nada que a primeira etapa já não tenha encontrado  
- ✔ Quase sempre retorna *“no improvement found”*  

👉 Conclusão: não agrega poder real à busca local.

---

### 2️⃣ `_one_insertion_first_improvement` está muito restrito

No seu código, o operador:

- evita `j == i`  
- evita `j == i + 1`  

Além disso, tenta reinserir usando **restrições para tour fechado/aberto** que acabam eliminando **70–80% das posições possíveis**.

**Problemas adicionais:**
- Você testa cada par `(i, j)`,  
- mas o cálculo de **delta frequentemente ignora casos adjacentes**,  
- fazendo com que muitos movimentos bons retornem `delta = 0` ou `delta > 0`.

👉 Resultado: o operador parece fraco, mas o problema está na implementação, não no conceito.

---

### 3️⃣ `_two_exchange_first_improvement` é redundante com insertion

O **2-exchange real** troca duas cidades, mas:

- para **TSP simétrico**,  
- o 2-swap quase nunca melhora,  
- é extremamente fraco como operador de descida local,  
- e é **dominado por insertion + 2-opt**.

**Resultado prático:**
- 🟣 quase sempre retorna *“no improvement”*  

👉 Em local search, esse operador não compensa.

---

## 🔥 O que é realmente forte em Local Search para TSP

Se você abrir qualquer solver **top-tier** (LKH, EAX, Helsgaun, HGS/VRP), verá que a descida local é construída em cima de:

**Operadores essenciais:**
- ✔ 2-opt  
- ✔ 3-opt  
- ✔ Or-opt (1-move, 2-move, 3-move)  
- ✔ Lin-Kernighan style 5-opt / k-opt dinâmico  
- ✔ Insertion (1-move) — **implementado corretamente**  
- ✔ Swap de 2 nós (**apenas para TSP assimétrico**)  

**Operadores que não aparecem:**
- ❌ Subsequence-flip limitado  
- ❌ 2-exchange puro  
- ❌ Double-bridge na local search (*usado apenas como shaking*)  

---

## ⭐ Or-opt completo (1-move, 2-move, 3-move)

Este é o operador **mais importante depois do 2-opt**.

Ele permite:
- mover **1 cidade**  
- mover **2 cidades contíguas**  
- mover **3 cidades contíguas**

💡 Com isso, você alcança **~95% da força do LKH**, sem a complexidade de um k-opt dinâmico completo.


In [ ]:
operators = [
    self._two_opt_first_improvement,
    self._three_opt_first_improvement,
    #self._or_opt_first_improvement,
    self._one_move_insertion_improvement,
    self._double_bridge_first_improvement,
]

In [ ]:
import numpy as np
from typing import Tuple

def _three_opt_first_improvement(
    self,
    tour: np.ndarray,
    current_distance: float,
    *,
    include_2opt_cases: bool = True,
    verify: bool = False,
    tol: float = 1e-12,
    k_neighbors: int = 20,   # typical 15–30
) -> Tuple[np.ndarray, bool, float]:
    """
    First-improvement 3-opt with candidate-list pruning and O(1) delta eval.
    Requires self._dist: np.ndarray[n, n] distances.
    Optional: self._nn nearest neighbors list (n x k); if absent, it will be built or we fallback.
    """
    is_closed = tour[0] == tour[-1]
    core = tour[:-1] if is_closed else tour
    n = len(core)
    if n < 6:
        return tour, False, current_distance

    dist = self._dist  # np.ndarray[n, n]

    # Build/ensure nearest neighbors (nn[u] is array of neighbor nodes for u)
    nn = getattr(self, "_nn", None)
    if nn is None or (len(nn) != n) or (len(nn[0]) < k_neighbors):
        nn = _build_candidate_lists(dist, k=k_neighbors)
        self._nn = nn

    # Helper to compute distance quickly
    def D(u: int, v: int) -> float:
        return dist[u, v]

    # Apply a selected 3-opt reconnection to the core tour
    def apply_3opt(core, i, j, k, variant_tag):
        # A = core[:i], B = core[i:j], C = core[j:k], D = core[k:]
        # Variants:
        #   2-opt equivalents (optional):
        #   "B^R C", "B C^R", "B^R C^R"
        #   True 3-opt:
        #   "C B", "C^R B", "C B^R", "C^R B^R"
        A = core[:i]
        B = core[i:j]
        C = core[j:k]
        D = core[k:]

        if variant_tag == "BrC":
            new_core = np.concatenate([A, B[::-1], C, D])
        elif variant_tag == "BCr":
            new_core = np.concatenate([A, B, C[::-1], D])
        elif variant_tag == "BrCr":
            new_core = np.concatenate([A, B[::-1], C[::-1], D])
        elif variant_tag == "CB":
            new_core = np.concatenate([A, C, B, D])
        elif variant_tag == "CrB":
            new_core = np.concatenate([A, C[::-1], B, D])
        elif variant_tag == "CBr":
            new_core = np.concatenate([A, C, B[::-1], D])
        elif variant_tag == "CrBr":
            new_core = np.concatenate([A, C[::-1], B[::-1], D])
        else:
            raise ValueError("Unknown variant")

        return (np.concatenate([new_core, new_core[:1]]) if is_closed else new_core)

    # ---- Main search with pruning by candidate lists ----
    # We iterate 'i', then try to pick 'j' and 'k' based on neighbor relationships
    # Heuristic: choose j so that core[j] (or core[j-1]) is in neighbors of core[i] (or core[i-1])
    # and choose k similarly with respect to core[j], to reduce search volume drastically.

    for i in range(1, n - 2):
        a = core[i - 1]
        b = core[i]

        # Consider j such that c=d's edge touches nearest neighbors of a or b
        # We will iterate j but skip most cases unless either core[j] or core[j-1]
        # is in the neighbor set of a or b (or vice-versa).
        a_neighbors = nn[a]
        b_neighbors = nn[b]

        for j in range(i + 1, n - 1):
            c = core[j - 1]
            d = core[j]

            # Quick neighbor gating
            if (d not in a_neighbors) and (d not in b_neighbors) and (c not in a_neighbors) and (c not in b_neighbors):
                continue

            for k in range(j + 1, n):
                e = core[k - 1]
                f = core[k]

                # Another neighbor gate w.r.t. j edge or the next edge (k)
                c_neighbors = nn[c]
                d_neighbors = nn[d]
                if (e not in c_neighbors) and (e not in d_neighbors) and (f not in c_neighbors) and (f not in d_neighbors):
                    continue

                # Ensure B, C, D non-empty (they are by loop ranges)
                # Removed sum:
                removed = D(a, b) + D(c, d) + D(e, f)

                # Prebind A_end and D_start
                A_end = a
                D_start = f

                # Evaluate variants by delta only, no array construction
                # We'll map each variant to the first and last of X and Y:
                # X, Y chosen from B or B^R and C or C^R
                B_first, B_last = b, c
                C_first, C_last = d, e

                # Helper to compute Δ for endpoints only
                def delta_for(x0, x1, y0, y1):
                    return (D(A_end, x0) + D(x1, y0) + D(y1, D_start)) - removed

                # Collect candidates (ordered to find good moves sooner)
                # 2-opt-equivalents (optional)
                if include_2opt_cases:
                    # "B^R C"  => X=B^R (x0=c, x1=b), Y=C (y0=d, y1=e)
                    d1 = delta_for(B_last, B_first, C_first, C_last)
                    if d1 < -tol:
                        new_tour = apply_3opt(core, i, j, k, "BrC")
                        return new_tour, True, (self._tour_length(new_tour) if verify else current_distance + d1)

                    # "B C^R" => X=B (x0=b, x1=c), Y=C^R (y0=e, y1=d)
                    d2 = delta_for(B_first, B_last, C_last, C_first)
                    if d2 < -tol:
                        new_tour = apply_3opt(core, i, j, k, "BCr")
                        return new_tour, True, (self._tour_length(new_tour) if verify else current_distance + d2)

                    # "B^R C^R" => X=B^R (c,b), Y=C^R (e,d)
                    d3 = delta_for(B_last, B_first, C_last, C_first)
                    if d3 < -tol:
                        new_tour = apply_3opt(core, i, j, k, "BrCr")
                        return new_tour, True, (self._tour_length(new_tour) if verify else current_distance + d3)

                # True 3‑opt cases (four)
                # "C B" => X=C (d,e), Y=B (b,c)
                d4 = delta_for(C_first, C_last, B_first, B_last)
                if d4 < -tol:
                    new_tour = apply_3opt(core, i, j, k, "CB")
                    return new_tour, True, (self._tour_length(new_tour) if verify else current_distance + d4)

                # "C^R B" => X=C^R (e,d), Y=B (b,c)
                d5 = delta_for(C_last, C_first, B_first, B_last)
                if d5 < -tol:
                    new_tour = apply_3opt(core, i, j, k, "CrB")
                    return new_tour, True, (self._tour_length(new_tour) if verify else current_distance + d5)

                # "C B^R" => X=C (d,e), Y=B^R (c,b)
                d6 = delta_for(C_first, C_last, B_last, B_first)
                if d6 < -tol:
                    new_tour = apply_3opt(core, i, j, k, "CBr")
                    return new_tour, True, (self._tour_length(new_tour) if verify else current_distance + d6)

                # "C^R B^R" => X=C^R (e,d), Y=B^R (c,b)
                d7 = delta_for(C_last, C_first, B_last, B_first)
                if d7 < -tol:
                    new_tour = apply_3opt(core, i, j, k, "CrBr")
                    return new_tour, True, (self._tour_length(new_tour) if verify else current_distance + d7)

    # No improvement
    return tour, False, current_distance


def _build_candidate_lists(dist: np.ndarray, k: int = 20):
    n = dist.shape[0]
    # argsort per row once; skip the 0 self-distance
    nn = []
    for u in range(n):
        # Sort neighbors by distance; take the first k excluding itself
        order = np.argsort(dist[u])
        order = order[order != u][:k]
        nn.append(order)
    return nn

In [52]:
from click import Tuple

def edge_cost(u: int, v: int, dist_matrix) -> float:
    lower, upper = (u, v) if u <= v else (v, u)
    return float(dist_matrix[lower, upper])

def tour_length(tour: np.ndarray, dist_matrix) -> float:
    """Works for open and explicitly closed tours (last == first)."""
    if len(tour) < 2:
        return 0.0
    total = 0.0
    if tour[0] == tour[-1]:
        # closed: sum edges between consecutive unique nodes
        for t in range(len(tour) - 1):
            total += edge_cost(tour[t], tour[t + 1], dist_matrix)
    else:
        # open: sum consecutive edges, no closure
        for t in range(len(tour) - 1):
            total += edge_cost(tour[t], tour[t + 1], dist_matrix)
    return total
    
def tour_distance(tour, dist_matrix):
    tour = np.asarray(tour, dtype=int)
    tour_shifted = np.roll(tour, -1)
    lower = np.minimum(tour, tour_shifted)
    upper = np.maximum(tour, tour_shifted)
    return float(np.sum(dist_matrix[lower, upper]))


import numpy as np
from typing import Tuple

def three_opt_first_improvement(
    tour: np.ndarray,
    current_distance: float,
    *,
    include_2opt_cases: bool = True,
    verify: bool = False,
    tol: float = 1e-12,
    k_neighbors: int = 20,   # typical 15–30
    distance_matrix: np.ndarray,
) -> Tuple[np.ndarray, bool, float]:
    """
    First-improvement 3-opt with candidate-list pruning and O(1) delta eval.
    Requires dist: np.ndarray[n, n] distances.
    Optional: self._nn nearest neighbors list (n x k); if absent, it will be built or we fallback.
    """
    is_closed = tour[0] == tour[-1]
    core = tour[:-1] if is_closed else tour
    n = len(core)
    if n < 6:
        return tour, False, current_distance

    dist = distance_matrix # np.ndarray[n, n]

    nn = _build_candidate_lists(dist, k=k_neighbors)

    # Helper to compute distance quickly
    def D(u: int, v: int) -> float:
        return dist[u, v]

    # Apply a selected 3-opt reconnection to the core tour
    def apply_3opt(core, i, j, k, variant_tag):
        # A = core[:i], B = core[i:j], C = core[j:k], D = core[k:]
        # Variants:
        #   2-opt equivalents (optional):
        #   "B^R C", "B C^R", "B^R C^R"
        #   True 3-opt:
        #   "C B", "C^R B", "C B^R", "C^R B^R"
        A = core[:i]
        B = core[i:j]
        C = core[j:k]
        D = core[k:]

        if variant_tag == "BrC":
            new_core = np.concatenate([A, B[::-1], C, D])
        elif variant_tag == "BCr":
            new_core = np.concatenate([A, B, C[::-1], D])
        elif variant_tag == "BrCr":
            new_core = np.concatenate([A, B[::-1], C[::-1], D])
        elif variant_tag == "CB":
            new_core = np.concatenate([A, C, B, D])
        elif variant_tag == "CrB":
            new_core = np.concatenate([A, C[::-1], B, D])
        elif variant_tag == "CBr":
            new_core = np.concatenate([A, C, B[::-1], D])
        elif variant_tag == "CrBr":
            new_core = np.concatenate([A, C[::-1], B[::-1], D])
        else:
            raise ValueError("Unknown variant")

        return (np.concatenate([new_core, new_core[:1]]) if is_closed else new_core)

    # ---- Main search with pruning by candidate lists ----
    # We iterate 'i', then try to pick 'j' and 'k' based on neighbor relationships
    # Heuristic: choose j so that core[j] (or core[j-1]) is in neighbors of core[i] (or core[i-1])
    # and choose k similarly with respect to core[j], to reduce search volume drastically.

    for i in range(1, n - 2):
        a = core[i - 1]
        b = core[i]

        # Consider j such that c=d's edge touches nearest neighbors of a or b
        # We will iterate j but skip most cases unless either core[j] or core[j-1]
        # is in the neighbor set of a or b (or vice-versa).
        a_neighbors = nn[a]
        b_neighbors = nn[b]

        for j in range(i + 1, n - 1):
            c = core[j - 1]
            d = core[j]

            # Quick neighbor gating
            if (d not in a_neighbors) and (d not in b_neighbors) and (c not in a_neighbors) and (c not in b_neighbors):
                continue

            for k in range(j + 1, n):
                e = core[k - 1]
                f = core[k]

                # Another neighbor gate w.r.t. j edge or the next edge (k)
                c_neighbors = nn[c]
                d_neighbors = nn[d]
                if (e not in c_neighbors) and (e not in d_neighbors) and (f not in c_neighbors) and (f not in d_neighbors):
                    continue

                # Ensure B, C, D non-empty (they are by loop ranges)
                # Removed sum:
                removed = D(a, b) + D(c, d) + D(e, f)

                # Prebind A_end and D_start
                A_end = a
                D_start = f

                # Evaluate variants by delta only, no array construction
                # We'll map each variant to the first and last of X and Y:
                # X, Y chosen from B or B^R and C or C^R
                B_first, B_last = b, c
                C_first, C_last = d, e

                # Helper to compute Δ for endpoints only
                def delta_for(x0, x1, y0, y1):
                    return (D(A_end, x0) + D(x1, y0) + D(y1, D_start)) - removed

                # Collect candidates (ordered to find good moves sooner)
                # 2-opt-equivalents (optional)
                if include_2opt_cases:
                    # "B^R C"  => X=B^R (x0=c, x1=b), Y=C (y0=d, y1=e)
                    d1 = delta_for(B_last, B_first, C_first, C_last)
                    if d1 < -tol:
                        new_tour = apply_3opt(core, i, j, k, "BrC")
                        return new_tour, True, (tour_length(new_tour, dist_matrix=dist) if verify else current_distance + d1)

                    # "B C^R" => X=B (x0=b, x1=c), Y=C^R (y0=e, y1=d)
                    d2 = delta_for(B_first, B_last, C_last, C_first)
                    if d2 < -tol:
                        new_tour = apply_3opt(core, i, j, k, "BCr")
                        return new_tour, True, (tour_length(new_tour, dist_matrix=dist) if verify else current_distance + d2)

                    # "B^R C^R" => X=B^R (c,b), Y=C^R (e,d)
                    d3 = delta_for(B_last, B_first, C_last, C_first)
                    if d3 < -tol:
                        new_tour = apply_3opt(core, i, j, k, "BrCr")
                        return new_tour, True, (tour_length(new_tour, dist_matrix=dist) if verify else current_distance + d3)

                # True 3‑opt cases (four)
                # "C B" => X=C (d,e), Y=B (b,c)
                d4 = delta_for(C_first, C_last, B_first, B_last)
                if d4 < -tol:
                    print("Improvement found: C B")
                    new_tour = apply_3opt(core, i, j, k, "CB")
                    print("New tour:", new_tour)
                    print(core)
                    
                    return new_tour, True, (tour_length(new_tour, dist_matrix=dist) if verify else current_distance + d4)

                # "C^R B" => X=C^R (e,d), Y=B (b,c)
                d5 = delta_for(C_last, C_first, B_first, B_last)
                if d5 < -tol:
                    new_tour = apply_3opt(core, i, j, k, "CrB")
                    return new_tour, True, (tour_length(new_tour, dist_matrix=dist) if verify else current_distance + d5)

                # "C B^R" => X=C (d,e), Y=B^R (c,b)
                d6 = delta_for(C_first, C_last, B_last, B_first)
                if d6 < -tol:
                    new_tour = apply_3opt(core, i, j, k, "CBr")
                    return new_tour, True, (tour_length(new_tour, dist_matrix=dist) if verify else current_distance + d6)

                # "C^R B^R" => X=C^R (e,d), Y=B^R (c,b)
                d7 = delta_for(C_last, C_first, B_last, B_first)
                if d7 < -tol:
                    new_tour = apply_3opt(core, i, j, k, "CrBr")
                    return new_tour, True, (tour_length(new_tour, dist_matrix=dist) if verify else current_distance + d7)

    # No improvement
    return tour, False, current_distance


def _build_candidate_lists(dist: np.ndarray, k: int = 20):
    n = dist.shape[0]
    # argsort per row once; skip the 0 self-distance
    nn = []
    for u in range(n):
        # Sort neighbors by distance; take the first k excluding itself
        order = np.argsort(dist[u])
        order = order[order != u][:k]
        nn.append(order)
    return nn


In [53]:
dist_matrix = np.random.rand(100, 100)
dist_matrix = (dist_matrix + dist_matrix.T) / 2  # Symmetric
np.fill_diagonal(dist_matrix, 0)


def random_tour():
    n = dist_matrix.shape[0]
    tour = np.arange(n)
    np.random.shuffle(tour)
    return np.append(tour, tour[0])


for _ in range(1000):
    tour = random_tour()
    d0 = tour_distance(tour, dist_matrix)

    new_tour, improved, d1 = three_opt_first_improvement(
        tour, d0, verify=True, distance_matrix=dist_matrix
    )

    if improved:
        assert abs(tour_distance(new_tour, dist_matrix) - d1) < 1e-9, "Distance mismatch after improvement"


Improvement found: C B
New tour: [11 41 33 39 82  8 47 28 55 63 27 57 74 71 84 24 76 66 29 54 51 88 31  7
 79 91 73 32 95 65 62 30  2 87 94 42 36  3 15 44 37 67 38 35 72  5 56 49
 78  4 16 46 18  0 93 64 45  9 14 77 22 21 25 81 92 75 58 48 60 43 17 89
 20 19 83 34 40 96 80 70 53 13 12 85 50 90 59 86 10 97 69 52  1 26 99 98
 23 68  6 61 11]
[11 33 41 39 82  8 47 28 55 63 27 57 74 71 84 24 76 66 29 54 51 88 31  7
 79 91 73 32 95 65 62 30  2 87 94 42 36  3 15 44 37 67 38 35 72  5 56 49
 78  4 16 46 18  0 93 64 45  9 14 77 22 21 25 81 92 75 58 48 60 43 17 89
 20 19 83 34 40 96 80 70 53 13 12 85 50 90 59 86 10 97 69 52  1 26 99 98
 23 68  6 61]
Improvement found: C B
New tour: [65 56 74  8 77 61 66 34 58 71 52  4 67 16 19 68 32 76 23 94 37 88 30 10
 17 50 24  3 99 53  5 86 98 60 64 41 79 45 82  7 13 18 11 27 63 55 90 81
 28 78  2 36 97 44 92 22 39 72 12 48 54 95 84 20 59 38 40 25  0 33 57 73
 29 83 43 93 70 96 75  6 69 87 21  9 31 49 46 47 51 26 80 15  1 85 62 89
 35 42 14 91 65]
[65 74  8 

In [30]:
def or_opt_first_improvement(tour: np.ndarray, current_distance: float, distance_matrix: np.ndarray):
        """
        OR-OPT first improvement para blocos de tamanho 1, 2 e 3.
        Implementação consistente com o cálculo de tour_distance.
        """
        is_closed = tour[0] == tour[-1]
        core = tour[:-1] if is_closed else tour
        n = len(core)

        def cost(a, b):
            lower, upper = (a, b) if a <= b else (b, a)
            return distance_matrix[lower, upper]

        # Tamanhos dos blocos OR-OPT
        for block_size in (1, 2, 3):
            if n <= block_size + 2:
                continue

            for i in range(1, n - block_size + 1):

                # bloco a ser movido
                block = core[i:i+block_size]

                prev_i = core[i - 1]
                next_i = core[(i + block_size) % n] if (is_closed or i + block_size < n) else None

                # custo da remoção
                removed = cost(prev_i, block[0])
                if next_i is not None:
                    removed += cost(block[-1], next_i)

                added = cost(prev_i, next_i) if next_i is not None else 0.0

                removal_delta = added - removed

                # vetor sem o bloco
                core_removed = np.delete(core, slice(i, i + block_size))
                m = len(core_removed)

                # tenta todos os destinos possíveis
                for j in range(1, m + 1):

                    # impedir movimentos triviais
                    if j == i or j == i + 1:
                        continue

                    prev_j = core_removed[j - 1] if j - 1 >= 0 else None
                    next_j = core_removed[j] if j < m else None

                    # custo da inserção
                    removed_ins = 0.0
                    if prev_j is not None and next_j is not None:
                        removed_ins += cost(prev_j, next_j)

                    added_ins = 0.0
                    if prev_j is not None:
                        added_ins += cost(prev_j, block[0])
                    if next_j is not None:
                        added_ins += cost(block[-1], next_j)

                    delta = removal_delta + (added_ins - removed_ins)

                    if delta < 0:
                        new_distance = current_distance + delta

                        # reconstrução CORRETA do tour
                        new_core = np.concatenate([core_removed[:j], block, core_removed[j:]])

                        if is_closed:
                            new_core = np.concatenate([new_core, [new_core[0]]])

                        return new_core, True, new_distance

        return tour, False, current_distance

In [35]:
dist_matrix = np.random.rand(100, 100)
dist_matrix = (dist_matrix + dist_matrix.T) / 2  # Symmetric
np.fill_diagonal(dist_matrix, 0)


def random_tour():
    n = dist_matrix.shape[0]
    tour = np.arange(n)
    np.random.shuffle(tour)
    return np.append(tour, tour[0])


for _ in range(1000):
    tour = random_tour()
    d0 = tour_distance(tour, dist_matrix)

    new_tour, improved, d1 = or_opt_first_improvement(
        tour, d0, distance_matrix=dist_matrix
    )
    

    if improved:
        print(tour, new_tour, tour_distance(new_tour, dist_matrix), d1)
        assert abs(tour_distance(new_tour, dist_matrix) - d1) < 1e-9, "Distance mismatch after improvement"


[63 75 49 76 30 73  6 13  5 93 82 40 17 19 95 91  4 92 39  8 11 18 23 25
 52 59 65 78 61 53 38 51 15 89 60  9 21 12 14  1 34 79 50 29 88 46 98 28
 36 56 80 90 47 94 70 99 68  3 44 16 71 42 87 67 84 20 66 96  2 54 43 10
 24  7 37 57 55 48 22 45 26 33 32  0 72 41 58 83 62 97 85 69 31 86 74 81
 35 27 77 64 63] [63 49 76 30 73  6 13  5 93 82 40 17 19 95 91  4 92 39  8 11 18 23 25 52
 59 65 78 61 53 38 51 15 89 60  9 21 12 14  1 34 79 50 29 88 46 98 28 36
 56 80 90 47 94 70 99 68  3 44 16 71 42 87 67 84 20 66 96  2 54 43 10 24
  7 37 57 55 48 22 45 26 33 32  0 72 41 58 83 62 97 85 69 75 31 86 74 81
 35 27 77 64 63] 49.4375536175657 49.43755361756569
[42 94 35 75 15 44 54 84 18 50 12 70 40 93 22  3 41 57 76 30 24 46  7 99
 11 71 97 62 39 51 13  8 10 85  5 68 28 98 25 73 45 17 29 89  0 80 74 81
 82 61 48 66 86 67  2 56 95 38 90 27 43 64  1 65  9 53 91 88 21 79 78 16
  6 23 49 52 31 14 69  4 83 59 19 60 55 58 87 72 96 20 32 77 37 92 47 26
 34 33 36 63 42] [42 35 75 94 15 44 54 84 18 50 12 70 4

In [38]:
def one_move_insertion_improvement(
    tour: np.ndarray,
    current_distance: float,
    *,
    verify: bool = False,
    tol: float = 1e-12,
    distance_matrix: np.ndarray,
) -> tuple[np.ndarray, bool, float]:

        is_closed = tour[0] == tour[-1]
        length = len(tour) - 1 if is_closed else len(tour)
        if length < 3:
            return tour, False, current_distance

        def edge_cost(u: int, v: int) -> float:
            lower, upper = (u, v) if u <= v else (v, u)
            return float(distance_matrix[lower, upper])

        core = tour[:-1] if is_closed else tour

        # i = posição do nó a mover (não mova a âncora core[0])
        for i in range(1, length):
            city = core[i]
            prev_i = core[i - 1]
            next_i = core[(i + 1) % length] if (is_closed or i + 1 < length) else None

            # Δ da remoção de 'city'
            print(f"Considering moving city {city} from position {i}")
            removed_rem = edge_cost(prev_i, city)
            print(f"  Removed edge cost: {prev_i} -> {city} = {removed_rem}")
            if next_i is not None:
                removed_rem += edge_cost(city, next_i)
            added_rem = edge_cost(prev_i, next_i) if next_i is not None else 0.0
            base_delta = added_rem - removed_rem
            print(f"added_rem: {added_rem}, removed_rem: {removed_rem}, base_delta: {base_delta}")

            # vetor sem 'city'
            core_removed = np.delete(core, i)
            m = len(core_removed)

            # j = posição de inserção (inserir antes de core_removed[j])
            for j in range(1, m + 1):
                # Observação: não precisamos pular j==i/i+1 aqui; se for no-op, delta=0.
                if is_closed:
                    # wrap-around correto para tour fechado
                    prev_j = core_removed[(j - 1) % m]
                    next_j = core_removed[j % m]
                else:
                    prev_j = core_removed[j - 1] if j - 1 >= 0 else None  # j começa em 1, então prev_j sempre existe
                    next_j = core_removed[j] if j < m else None            # se j==m, insere ao final (sem next_j)

                # Δ da inserção entre (prev_j, next_j) → (prev_j, city) + (city, next_j)
                removed_ins = edge_cost(prev_j, next_j) if (prev_j is not None and next_j is not None) else 0.0
                added_ins = 0.0
                if prev_j is not None:
                    added_ins += edge_cost(prev_j, city)
                if next_j is not None:
                    added_ins += edge_cost(city, next_j)

                delta = base_delta + (added_ins - removed_ins)

                print(f"  Trying to insert at position {j}:")
                print(f"    removed_ins: {removed_ins}, added_ins: {added_ins}, delta: {delta}")

                if delta < -tol:
                    # Reconstrução consistente com o delta
                    new_core = np.concatenate([core_removed[:j], [city], core_removed[j:]])
                    candidate = (
                        np.concatenate([new_core, new_core[:1]]) if is_closed else new_core
                    )
                    new_distance = current_distance + delta

                    if verify:
                        pass

                    else:
                        return candidate, True, new_distance

        return tour, False, current_distance
    

In [43]:
import numpy as np
dist_matrix = np.random.rand(100, 100)
dist_matrix = (dist_matrix + dist_matrix.T) / 2  # Symmetric
np.fill_diagonal(dist_matrix, 0)


def random_tour():
    n = dist_matrix.shape[0]
    tour = np.arange(n)
    np.random.shuffle(tour)
    return np.append(tour, tour[0])


for _ in range(1000):
    tour = random_tour()
    d0 = tour_distance(tour, dist_matrix)

    new_tour, improved, d1 = one_move_insertion_improvement(
        tour, d0, distance_matrix=dist_matrix
    )
    

    if improved:
        print(tour, new_tour, tour_distance(new_tour, dist_matrix), d1)
        assert abs(tour_distance(new_tour, dist_matrix) - d1) < 1e-9, "Distance mismatch after improvement"


Considering moving city 79 from position 1
  Removed edge cost: 6 -> 79 = 0.6136992080236335
added_rem: 0.7860423164287499, removed_rem: 0.9793144137230725, base_delta: -0.1932720972943226
  Trying to insert at position 1:
    removed_ins: 0.7860423164287499, added_ins: 0.9793144137230725, delta: 0.0
  Trying to insert at position 2:
    removed_ins: 0.24957878055848226, added_ins: 1.0562482019496289, delta: 0.613397324096824
  Trying to insert at position 3:
    removed_ins: 0.613239951190178, added_ins: 1.3809332006495225, delta: 0.5744211521650219
  Trying to insert at position 4:
    removed_ins: 0.21908031108121556, added_ins: 1.4092296715073411, delta: 0.9968772631318029
  Trying to insert at position 5:
    removed_ins: 0.897974699908744, added_ins: 1.4205443864128533, delta: 0.3292975892097867
  Trying to insert at position 6:
    removed_ins: 0.1987984143222553, added_ins: 1.2784160243026943, delta: 0.8863455126861163
  Trying to insert at position 7:
    removed_ins: 0.495653